# Wikipedia Exploration: BFS vs DFS

This notebook runs the same starting page through two different traversal strategies and shows how the choice of data structure (Queue vs Stack) completely changes what gets discovered.

**Goal:** Explore up to N pages **or** X depth-hops from `"Data science"`, then compare what BFS and DFS each found.

| Strategy | Frontier | Behaviour |
|---|---|---|
| BFS | Queue (FIFO) | Layer-by-layer — stays near the start topic |
| DFS | Stack (LIFO) | Dives deep — reaches unrelated topics quickly |

In [ ]:
import sys
sys.path.insert(0, "..")

from src import BFSFrontier, DFSFrontier, WikiScraper, WikiExplorer
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Configuration

Adjust `START_PAGE`, `MAX_PAGES`, and `MAX_DEPTH` here. The explorer stops at whichever limit is hit first.

- `MAX_DEPTH = 1` → only direct links from the start page
- `MAX_DEPTH = 2` → direct links + their links
- `MAX_DEPTH = None` → no depth limit, only page count

In [ ]:
START_PAGE = "Data science"
MAX_PAGES  = 30
MAX_DEPTH  = 2       # set to None to remove the depth limit

SCRAPER = WikiScraper(delay=0.3)

## Run BFS exploration

In [ ]:
print("=" * 55)
print("BFS  (Queue / FIFO / layer-by-layer)")
print("=" * 55)

bfs_explorer = WikiExplorer(
    BFSFrontier(), scraper=SCRAPER,
    max_pages=MAX_PAGES, max_depth=MAX_DEPTH, verbose=True
)
bfs_graph = bfs_explorer.explore(START_PAGE)
print(f"\nBFS result: {bfs_graph}")

## Run DFS exploration

In [ ]:
print("=" * 55)
print("DFS  (Stack / LIFO / deep-dive)")
print("=" * 55)

dfs_explorer = WikiExplorer(
    DFSFrontier(), scraper=SCRAPER,
    max_pages=MAX_PAGES, max_depth=MAX_DEPTH, verbose=True
)
dfs_graph = dfs_explorer.explore(START_PAGE)
print(f"\nDFS result: {dfs_graph}")

## Side-by-side comparison

In [ ]:
bfs_only = bfs_graph.visited - dfs_graph.visited
dfs_only = dfs_graph.visited - bfs_graph.visited
shared   = bfs_graph.visited & dfs_graph.visited

print(f"Pages visited by BOTH    : {len(shared)}")
print(f"Pages visited by BFS only: {len(bfs_only)}")
print(f"  → {sorted(bfs_only)}")
print(f"Pages visited by DFS only: {len(dfs_only)}")
print(f"  → {sorted(dfs_only)}")
print()
print("BFS stays close to the start topic (mostly data/science pages).")
print("DFS wanders — its unique pages will look unrelated to data science.")

## Visualise BFS subgraph — radial / wide

In [ ]:
def build_nx_graph(wiki_graph, limit=25):
    G = nx.DiGraph()
    visited_list = list(wiki_graph.visited)[:limit]
    for source in visited_list:
        for target in wiki_graph.neighbors(source):
            if target in wiki_graph.visited:
                G.add_edge(source, target)
    return G

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for ax, graph, title, color in [
    (axes[0], bfs_graph, f"BFS — '{START_PAGE}'", "#4DB8FF"),
    (axes[1], dfs_graph, f"DFS — '{START_PAGE}'", "#2ecc71"),
]:
    G = build_nx_graph(graph)
    pos = nx.spring_layout(G, seed=42, k=0.9)
    node_colors = ["#e74c3c" if n == START_PAGE else color for n in G.nodes()]
    nx.draw_networkx(
        G, pos, ax=ax,
        node_color=node_colors, node_size=500,
        font_size=6, arrows=True, edge_color="#555", width=0.5,
    )
    ax.set_title(title, fontsize=13)
    ax.axis("off")

plt.suptitle(
    f"Same start page, same code, different frontier → different knowledge map\n"
    f"(max_pages={MAX_PAGES}, max_depth={MAX_DEPTH})",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("exploration_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## What the depth limit does

Re-run the cells below with different `max_depth` values to see how the exploration changes:

| `max_depth` | What gets visited |
|---|---|
| `1` | Only direct links from the start page |
| `2` | Direct links + one more hop |
| `3` | Three hops out — concepts start becoming less related |
| `None` | No depth limit — only the page count cap applies |

For BFS this is intuitive: depth 1 = layer 1, depth 2 = layers 1+2.  
For DFS the depth limit cuts off the dive at a fixed number of hops, preventing it from going arbitrarily far in one direction.

In [ ]:
# Depth experiment: BFS at depth=1 vs depth=2
for depth in [1, 2]:
    exp = WikiExplorer(
        BFSFrontier(), scraper=SCRAPER,
        max_pages=100, max_depth=depth, verbose=False
    )
    g = exp.explore(START_PAGE)
    print(f"BFS depth={depth}: {g.node_count()} pages visited")
    print(f"  Unique pages: {sorted(g.visited)[:8]} ...\n")